# promote

Carries Saved tables from your Dev Database to your Prod Database. Run each step's cell in turn, or, in their place, the last cell, project.run_workflow(...), which runs every step left in turn, up to the first only you can do, and says where it stopped. First fill in the names cell below and save the notebook: each call reads it, and so do project.check(), which names the next step, and project.do_next(), the cell before the last, which runs only the steps that change files, up to the first it can't.

In [ ]:
import pathlib
import sys
sys.path.insert(0, str(pathlib.Path.cwd().parents[2]))  # the Project: three up

from sqlglot_composer import admin, hive_settings, project, run_execute, run_query, show_hive
from warehouse import query, execute

In [ ]:
TABLES = []  # the dev tables, by their names in your Dev Database, such as ["mart_dev.daily_job_runs"]

In [ ]:
DECISIONS = {}  # your answer to each open Decision, by the table's name as
# project.check() writes it, such as
# {"mart_dev.weekly_ops": {"location": "/data/team/mart_dev.db/weekly",
#                          "key": ["job_id"], "types": {"fails": "bigint"}}}
# "key": None says no columns name one row. project.check() names each one open;
# a Key project.convert proposed is made, but an answer here changes it

## 1. Promote each table: a prod copy of its folder and Table reference

It changes only files, so project.do_next() runs it for you.

In [ ]:
project.promote(TABLES)

## 2. Answer each prod table's open Decisions in the decisions cell

It is a decision, yours to make.

## 3. Write each answer into its prod table's Table reference

It changes only files, so project.do_next() runs it for you.

In [ ]:
project.write_decisions(DECISIONS)

## 4. Create each table in prod

It is a call to the warehouse, yours to run.

In [ ]:
for table in project.promoted_tables(TABLES):
    run_execute(project.load(table).create(), execute)

## 5. Seed each prod table that reads itself, once, from its original

It is a call to the warehouse, yours to run.

In [ ]:
run_execute(hive_settings("dynamic_partitions"), execute)
for table in project.promoted_tables(TABLES, seeded=True):
    run_execute(project.load(table).seed(), execute)

## 6. Write each table in prod

It is a call to the warehouse, yours to run.

In [ ]:
project.run_tables(tables=project.promoted_tables(TABLES), execute=execute)

In [ ]:
project.do_next()

In [ ]:
# Every step left, in place of their cells, up to the first only you can do:
project.run_workflow("promote", query=query, execute=execute)